# Exercício 7 — Cobertura de Ruas com Sinalizadores

Um parque industrial possui 11 ruas (A a K) e 8 possíveis locais de instalação de sinalizadores (1 a 8).  
O objetivo é instalar o **mínimo de sinalizadores** de forma que **toda rua** seja atendida por ao menos um sinalizador.

**Cobertura por localização** (baseada na figura do mapa):

| Local | Ruas cobertas       |
|-------|---------------------|
| 1     | A, G                |
| 2     | A, B, F, I          |
| 3     | B, C, K             |
| 4     | F, H, I             |
| 5     | C, J, K             |
| 6     | E, F, G             |
| 7     | D, E, H             |
| 8     | D, J                |

**Formulação (Cobertura de Conjuntos — Set Cover ILP):**

$$\min \sum_{i=1}^{8} x_i$$

Sujeito a:
$$\sum_{i \in S_r} x_i \geq 1, \quad \forall r \in \{A, B, \ldots, K\}$$
$$x_i \in \{0, 1\}, \quad i = 1, \ldots, 8$$

onde $S_r$ é o conjunto de locais que cobrem a rua $r$.

In [1]:
!pip install ortools -q

In [2]:
from ortools.linear_solver import pywraplp

# Ruas do parque
ruas = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']

# Locais de instalação e quais ruas cada um cobre
cobertura = {
    1: ['A', 'G'],
    2: ['A', 'B', 'F', 'I'],
    3: ['B', 'C', 'K'],
    4: ['F', 'H', 'I'],
    5: ['C', 'J', 'K'],
    6: ['E', 'F', 'G'],
    7: ['D', 'E', 'H'],
    8: ['D', 'J'],
}
locais = list(cobertura.keys())

In [3]:
solver = pywraplp.Solver.CreateSolver('SCIP')
if solver is None:
    raise RuntimeError('Não foi possível criar o solver SCIP.')

# Variáveis binárias: x[i] = 1 se o sinalizador é instalado no local i
x = {i: solver.BoolVar(f'x{i}') for i in locais}

In [4]:
# Função objetivo: minimizar número de sinalizadores instalados
objetivo = solver.Objective()
for i in locais:
    objetivo.SetCoefficient(x[i], 1)
objetivo.SetMinimization()

In [5]:
# Restrições de cobertura: cada rua deve ser coberta por ao menos um sinalizador
for r in ruas:
    locais_que_cobrem = [i for i in locais if r in cobertura[i]]
    ct = solver.RowConstraint(1, solver.infinity(), f'cobre_{r}')
    for i in locais_que_cobrem:
        ct.SetCoefficient(x[i], 1)

In [6]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 11
\   Variables        : 8
\     Binary         : 8
\     Integer        : 0
\     Continuous     : 0
Minimize
 Obj: +1 x1 +1 x2 +1 x3 +1 x4 +1 x5 +1 x6 +1 x7 +1 x8 
Subject to
 cobre_A: +1 x1 +1 x2  >= 1
 cobre_B: +1 x2 +1 x3  >= 1
 cobre_C: +1 x3 +1 x5  >= 1
 cobre_D: +1 x7 +1 x8  >= 1
 cobre_E: +1 x6 +1 x7  >= 1
 cobre_F: +1 x2 +1 x4 +1 x6  >= 1
 cobre_G: +1 x1 +1 x6  >= 1
 cobre_H: +1 x4 +1 x7  >= 1
 cobre_I: +1 x2 +1 x4  >= 1
 cobre_J: +1 x5 +1 x8  >= 1
 cobre_K: +1 x3 +1 x5  >= 1
Bounds
 0 <= x1 <= 1
 0 <= x2 <= 1
 0 <= x3 <= 1
 0 <= x4 <= 1
 0 <= x5 <= 1
 0 <= x6 <= 1
 0 <= x7 <= 1
 0 <= x8 <= 1
Binaries
 x1
 x2
 x3
 x4
 x5
 x6
 x7
 x8
End



In [7]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    z_otimo = int(objetivo.Value())
    solucao = [i for i in locais if x[i].solution_value() > 0.5]

    print(f'Número mínimo de sinalizadores: {z_otimo}')
    print(f'Locais selecionados: {solucao}')
    print()
    print('Verificação de cobertura:')
    for r in ruas:
        coberta_por = [i for i in solucao if r in cobertura[i]]
        print(f'  Rua {r}: coberta pelo(s) local(is) {coberta_por}')
else:
    print('Modelo sem solução ótima.')

Número mínimo de sinalizadores: 4
Locais selecionados: [2, 5, 6, 7]

Verificação de cobertura:
  Rua A: coberta pelo(s) local(is) [2]
  Rua B: coberta pelo(s) local(is) [2]
  Rua C: coberta pelo(s) local(is) [5]
  Rua D: coberta pelo(s) local(is) [7]
  Rua E: coberta pelo(s) local(is) [6, 7]
  Rua F: coberta pelo(s) local(is) [2, 6]
  Rua G: coberta pelo(s) local(is) [6]
  Rua H: coberta pelo(s) local(is) [7]
  Rua I: coberta pelo(s) local(is) [2]
  Rua J: coberta pelo(s) local(is) [5]
  Rua K: coberta pelo(s) local(is) [5]
